# Grok-robotics-08-mbrl-mpc

**Stage 08 — Model-Based RL & MPC**

## 概念
样本效率关键路径：
1. **学动力学模型** f̂(s,a) → s'
2. **在模型里规划**（随机射击 / CEM）
3. **Dyna**：用模型生成虚拟经验更新价值

对比纯 model-free：同样交互步数下，规划往往更快修好摆。


In [ ]:

import json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED=0; np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu)

class Pendulum:
    def __init__(self):
        self.max_speed=8.; self.max_torque=2.; self.dt=0.05; self.g=10.; self.m=1.; self.l=1.
        self.reset()
    def reset(self):
        self.state=np.array([np.random.uniform(-np.pi,np.pi), np.random.uniform(-1,1)],np.float32); return self.obs()
    def obs(self):
        th,thdot=self.state; return np.array([np.cos(th),np.sin(th),thdot],np.float32)
    def step(self,u):
        th,thdot=self.state; u=float(np.clip(u,-self.max_torque,self.max_torque))
        cost=(((th+np.pi)%(2*np.pi))-np.pi)**2 + 0.1*thdot**2 + 0.001*u**2
        newthdot=thdot+(-3*self.g/(2*self.l)*np.sin(th+np.pi)+3./(self.m*self.l**2)*u)*self.dt
        newth=th+newthdot*self.dt; newthdot=np.clip(newthdot,-self.max_speed,self.max_speed)
        self.state=np.array([newth,newthdot],np.float32); return self.obs(), -cost, False, {}

class DynModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(4,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,3))
    def forward(self,s,a):
        return self.net(torch.cat([s,a],-1))

def collect_random(env, n=3000):
    data=[]
    s=env.reset()
    for _ in range(n):
        a=np.array([np.random.uniform(-2,2)],np.float32)
        ns,r,_,_=env.step(a[0]); data.append((s,a,r,ns)); s=ns
        if len(data)%200==0: s=env.reset()
    return data

def train_model(data, epochs=30):
    m=DynModel().to(device)
    opt=torch.optim.Adam(m.parameters(), lr=1e-3)
    S=torch.tensor(np.array([d[0] for d in data]),device=device)
    A=torch.tensor(np.array([d[1] for d in data]),device=device)
    NS=torch.tensor(np.array([d[3] for d in data]),device=device)
    for _ in range(epochs):
        pred=m(S,A)
        loss=F.mse_loss(pred, NS)
        opt.zero_grad(); loss.backward(); opt.step()
    return m, float(loss.item())

def reward_fn(s,a):
    # s: cos,sin,thdot
    # recover th approx via atan2
    th=torch.atan2(s[...,1], s[...,0])
    thdot=s[...,2]
    return -(th**2 + 0.1*thdot**2 + 0.001*a.squeeze(-1)**2)

def mpc_action(model, s, horizon=12, pop=200):
    # random shooting
    s0=torch.tensor(s,device=device).float()
    acts=torch.empty(pop,horizon,1,device=device).uniform_(-2,2)
    st=s0.unsqueeze(0).repeat(pop,1)
    rets=torch.zeros(pop,device=device)
    for t in range(horizon):
        a=acts[:,t]
        rets += reward_fn(st,a)
        st = model(st,a)
    best=int(torch.argmax(rets).item())
    return float(acts[best,0,0].item())

def eval_controller(controller, n=8, T=200):
    env=Pendulum(); scores=[]
    for _ in range(n):
        s=env.reset(); R=0
        for _ in range(T):
            a=controller(s); s,r,_,_=env.step(a); R+=r
        scores.append(R)
    return float(np.mean(scores))


In [ ]:

t0=time.time()
env=Pendulum()
data=collect_random(env, n=4000)
model, final_loss = train_model(data, epochs=40)
print("model loss", final_loss)

rand_score=eval_controller(lambda s: float(np.random.uniform(-2,2)))
mpc_score=eval_controller(lambda s: mpc_action(model,s))
# true-model MPC oracle for upper bound (uses real step for planning? skip) 
elapsed=time.time()-t0
print("random", rand_score, "learned-model MPC", mpc_score)

# learning curve proxy: more data -> better MPC
curve=[]
for n in [500,1000,2000,4000]:
    d=data[:n]
    m,_=train_model(d, epochs=25)
    sc=eval_controller(lambda s, m=m: mpc_action(m,s), n=5)
    curve.append({"n":n, "score":sc}); print(curve[-1])

fig,ax=plt.subplots(figsize=(7,4))
ax.plot([c["n"] for c in curve],[c["score"] for c in curve], marker="o", label="MPC with learned model")
ax.axhline(rand_score, ls="--", c="gray", label="random")
ax.set_xlabel("# transitions for model"); ax.set_ylabel("eval return"); ax.legend()
ax.set_title("Model-based MPC sample efficiency")
fig.tight_layout(); fig.savefig(OUT/"stage08_mbrl_mpc.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage":"08-mbrl-mpc",
  "title":"Grok-robotics-08-mbrl-mpc",
  "metrics":{"random": rand_score, "mpc": mpc_score, "model_loss": final_loss, "curve": curve},
  "gpu": gpu, "elapsed_sec": elapsed,
  "concept": "learn dynamics then plan (random-shooting MPC)",
  "new_capability": "sample-efficient control via model + planning instead of pure trial policy gradient",
  "compare_to_previous": "Stage07 SAC is model-free continuous; Stage08 reuses data to build f̂ and plan",
}
assert payload["metrics"]["mpc"] > payload["metrics"]["random"]
(OUT/"results_stage08.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE08_OK")
